# 🌐 Distributed Systems Fundamentals — The Master Guide
### *From Zero to Interview-Ready*

---

> **Mental Model First:**
> A distributed system is a team of workers who can't share a whiteboard. Each worker has their own notebook. Coordinating them is the hard part. CAP Theorem says: when the phone line between workers goes down (partition), you must choose — do you refuse to answer until you've synced with everyone (consistency), or do you give your best answer from your own notebook (availability)? Consistent hashing is the seating chart: stable assignments that require minimal reshuffling when someone joins or leaves. Replication is keeping multiple notebooks in sync. Leader election decides who holds the authoritative notebook.

---

## 📋 Table of Contents

| # | Section |
|---|----------|
| 1 | [What Is Distributed Systems? The Visual Model](#1) |
| 2 | [Creating / Setup — Core Abstractions](#2) |
| 3 | [The Core API — Consistency, Availability, Partition Tolerance](#3) |
| 4 | [Decision Map — When to Prioritize What](#4) |
| 5 | [Pattern 1: CAP Theorem & Consistency Models](#5) |
| 6 | [Pattern 2: Consistent Hashing](#6) |
| 7 | [Pattern 3: Replication Strategies](#7) |
| 8 | [Pattern 4: Partitioning / Sharding](#8) |
| 9 | [Pattern 5: Distributed Consensus & Leader Election](#9) |
| 10 | [The Distributed Systems Decision Map](#10) |
| 11 | [Interview Cheat Sheet](#11) |
| 12 | [Summary Map](#12) |

<a id='1'></a>
## 1. 🗺️ What Is Distributed Systems? The Visual Model

```
CAP THEOREM — choose 2 of 3:

          Consistency
              △
             /|\
            / | \
           /  |  \
          /   |   \
  CP ────/ HBase, \──── CA
        / Zookeeper \  (not possible
       /  Mongo(w)  \  in practice —
      /              \  network always
     ◁────────────────▷  partitions)
  Partition         Availability
  Tolerance    AP: Cassandra,
               DynamoDB, Kafka

  REAL CHOICE: When a partition occurs (network split),
    CP: reject requests until partition heals → consistent, but unavailable
    AP: serve stale data or accept writes → available, but temporarily inconsistent

CONSISTENT HASHING RING:

  Hash space: 0 ─────────────────── 2^32 (wrap around like a clock)
                    Node A (hash=100)
         ╭──────────────────────────╮
        /   Node C (hash=250)        \
       │                             │  Key K (hash=120) → Node A
       │                             │  Key M (hash=220) → Node C
        \   Node B (hash=180)        /   Key Z (hash=50)  → Node A (wraps)
         ╰──────────────────────────╯

  Add Node D at hash=150: only keys in (100, 150] reassigned D → C
  Remove Node A: only its keys reassigned to next node (Node C)
  Virtual nodes: each physical node = 100-150 virtual positions → even load
```

<a id='2'></a>
## 2. 🔧 Creating / Setup — Core Abstractions

In [ ]:
# DISTRIBUTED SYSTEMS CORE ABSTRACTIONS

import hashlib
import bisect
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Any

@dataclass
class Node:
    node_id: str
    host: str
    port: int
    is_leader: bool = False
    data: Dict = field(default_factory=dict)   # local state
    is_healthy: bool = True

    def write(self, key: str, value: Any):
        self.data[key] = value

    def read(self, key: str) -> Optional[Any]:
        return self.data.get(key)

def hash_key(key: str) -> int:
    """MD5 hash → integer in [0, 2^32).  Used for consistent hashing."""
    return int(hashlib.md5(key.encode()).hexdigest(), 16) % (2**32)

# Consistency models (from weakest to strongest)
CONSISTENCY_MODELS = {
    "eventual": "All replicas will agree, eventually. No ordering guarantee.",
    "read_your_writes": "After you write, you will always read your own write.",
    "monotonic_read": "If you read X, future reads will never return older values.",
    "causal": "Operations causally related appear in order everywhere.",
    "sequential": "All nodes see all writes in the same order.",
    "linearizable": "Strongest: reads always reflect latest committed write (like a single node).",
}

print("Consistency Models (weakest to strongest):")
for i, (model, desc) in enumerate(CONSISTENCY_MODELS.items(), 1):
    print(f"  {i}. {model:20s}: {desc}")

# ACID vs BASE
print()
print("ACID (OLTP / relational DB):")
for prop, meaning in [("Atomicity","all-or-nothing transaction"),("Consistency","DB invariants always hold"),("Isolation","concurrent txns don't interfere"),("Durability","committed writes survive crashes")]:
    print(f"  {prop}: {meaning}")

print()
print("BASE (NoSQL / distributed):")
for prop, meaning in [("Basically Available","system stays up even during failures"),("Soft state","state may change without input (convergence)"),("Eventual consistency","replicas will agree, eventually")]:
    print(f"  {prop}: {meaning}")

<a id='3'></a>
## 3. ⚡ The Core API — Consistency, Availability, Partition Tolerance

```
CONCEPT              DEFINITION                          REAL SYSTEMS
─────────────────────────────────────────────────────────────────────────────
Consistency (C)      Every read returns the latest write  HBase, ZooKeeper
Availability (A)     Every request gets a response        Cassandra, DynamoDB
Partition Tol. (P)   System works despite network splits  ALL production systems
─────────────────────────────────────────────────────────────────────────────
Replication factor   N copies of each data item           N=3 typical
Write quorum W       Must ack from W nodes to confirm     W=2 for N=3
Read quorum R        Must read from R nodes and take latest R=2 for N=3
Strong consistency:  W + R > N  (quorums overlap, always see latest write)
─────────────────────────────────────────────────────────────────────────────

QUORUM MATH (N=3):
  W=3, R=1: strong consistency, low read cost, high write cost (all must ack)
  W=1, R=3: strong consistency, high read cost, low write cost
  W=2, R=2: strong consistency (2+2>3), balanced — most common
  W=1, R=1: eventual consistency — fastest, weakest guarantee

THINGS YOU DO NOT DO:
❌  Say "CA system" — CA is impossible when P is required (network always partitions)
✅  Frame as: "we choose CP or AP when a partition occurs"
❌  Use linearizable consistency for all data — extremely expensive at scale
✅  Choose the weakest consistency that satisfies the use case
❌  One shard for everything — hotspot problem
✅  Hash or range shard with awareness of access patterns
```

In [ ]:
# LIVE DEMO: quorum writes and reads with N=3 nodes

from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple

class QuorumCluster:
    def __init__(self, n: int = 3, w: int = 2, r: int = 2):
        self.n = n                      # replication factor
        self.w = w                      # write quorum
        self.r = r                      # read quorum
        self.nodes = [Node(f"node{i}", f"host{i}", 9000+i) for i in range(n)]
        print(f"Cluster: N={n}, W={w}, R={r}  consistency={'strong' if w+r>n else 'eventual'}")

    def write(self, key: str, value: Any, simulate_failure: int = None) -> bool:
        acks = 0
        for i, node in enumerate(self.nodes):
            if i == simulate_failure:
                print(f"  node{i}: FAILED (simulated)")
                continue
            node.write(key, value)
            acks += 1
            print(f"  node{i}: ACK")

        if acks >= self.w:
            print(f"  WRITE SUCCESS: {acks}/{self.n} acked (quorum={self.w})")
            return True
        else:
            print(f"  WRITE FAILED: only {acks}/{self.n} acked (need {self.w})")
            return False

    def read(self, key: str, simulate_failure: int = None) -> Optional[Any]:
        responses = []
        for i, node in enumerate(self.nodes):
            if i == simulate_failure:
                print(f"  node{i}: FAILED (simulated)")
                continue
            val = node.read(key)
            responses.append(val)
            print(f"  node{i}: returned {repr(val)}")

        if len(responses) >= self.r:
            # take the latest — in real systems this uses vector clocks or timestamps
            result = max((v for v in responses if v is not None), default=None)
            print(f"  READ SUCCESS: quorum of {len(responses)} → returning {repr(result)}")
            return result
        print(f"  READ FAILED: only {len(responses)} responses (need {self.r})")
        return None

print("=== Write with one node failure ===")
cluster = QuorumCluster(n=3, w=2, r=2)
cluster.write("user:alice", {"email": "alice@example.com"}, simulate_failure=2)
print()
print("=== Read (one node failed) ===")
cluster.read("user:alice", simulate_failure=0)  # node0 failed, node1+node2 respond

<a id='4'></a>
## 4. 🗂️ Decision Map — When to Prioritize What

```
USE CASE                           CONSISTENCY    SYSTEM CHOICE
──────────────────────────────────────────────────────────────────────
Banking / financial transactions    CP (strong)    PostgreSQL, HBase
User sessions / shopping cart       AP (eventual)  DynamoDB, Cassandra
Social media feed / recommendations AP (eventual)  Cassandra, Redis
Configuration / distributed locks   CP (strong)    ZooKeeper, etcd
Metrics / analytics                 AP (eventual)  Kafka, ClickHouse
Search index                        AP + near-real Elasticsearch
──────────────────────────────────────────────────────────────────────

REPLICATION STRATEGY:
  Primary-Replica:   one writer (primary), N readers (replicas)
                     simple, consistent writes, replica lag possible
  Multi-Primary:     any node accepts writes, resolve conflicts
                     high availability, conflict resolution needed
  Leaderless (Dynamo): any node accepts, quorum determines winner
                     highest availability, eventual consistency

SHARDING STRATEGY:
  Range sharding:    keys 0-999 → shard1, 1000-1999 → shard2, ...
                     good for range queries, hotspot risk (sequential writes)
  Hash sharding:     hash(key) % N → shard#
                     even distribution, poor range queries
  Consistent hashing: ring hash, virtual nodes
                     minimal reshuffling on node add/remove
  Directory-based:   lookup table maps key → shard
                     flexible, single point of failure (the directory)
```

<a id='5'></a>
## 5. 🧩 Pattern 1: CAP Theorem & Consistency Models

---

```
SCENARIO:  Two data centers (DC1, DC2) with replicated user database.
           Network link between DCs goes down (partition). A user updates
           their email in DC1. Another user tries to read it from DC2.

CP CHOICE (Consistency over Availability):
  DC2 refuses to serve reads for this user until the partition heals.
  Returns: 503 Service Unavailable
  Reason: returning stale data would be wrong (e.g., billing system)
  Use cases: financial data, inventory counts, distributed locks

AP CHOICE (Availability over Consistency):
  DC2 returns the stale email address (the old one).
  Returns: old email — user sees inconsistent data temporarily
  Eventually: when partition heals, DCs sync and converge
  Use cases: user profiles, shopping carts, social feeds

CONSISTENCY SPECTRUM (weakest to strongest):

  Eventual ──────────────────────────────────► Linearizable
  (DNS, social feeds)         (bank accounts, distributed locks)
  Fast, available              Slow, CP, serialized

  Cassandra tunable: ONE / QUORUM / ALL
    ONE:    fastest, weakest (any replica responds)
    QUORUM: R+W > N, strong consistency
    ALL:    every replica must respond — slowest, strongest

CONFLICT RESOLUTION (for AP systems):
  Last Write Wins (LWW):    take the write with the latest timestamp
  Version vectors:          detect concurrent conflicting writes, surface to client
  CRDTs:                    data structures that merge deterministically
                            (counters, sets, maps with defined semantics)
```

In [ ]:
# CAP THEOREM SIMULATION: CP vs AP during a network partition

from dataclasses import dataclass, field
from typing import Optional, Dict, Any
import time

@dataclass
class Replica:
    name: str
    data: Dict[str, Any] = field(default_factory=dict)
    timestamps: Dict[str, float] = field(default_factory=dict)
    reachable: bool = True   # False = partitioned from others

    def write(self, key, value):
        self.data[key] = value
        self.timestamps[key] = time.time()

    def read(self, key):
        return self.data.get(key)

class CPDatabase:
    """CP: consistency over availability. Rejects reads during partition."""
    def __init__(self, replicas):
        self.replicas = replicas
        self.partitioned = False

    def write(self, key, value):
        if self.partitioned:
            reachable = [r for r in self.replicas if r.reachable]
            if len(reachable) < len(self.replicas):
                print(f"[CP] WRITE REFUSED: partition detected, can't reach quorum")
                return False
        for r in self.replicas:
            r.write(key, value)
        print(f"[CP] WRITE OK: {key}={value} on all {len(self.replicas)} replicas")
        return True

    def read(self, key):
        reachable = [r for r in self.replicas if r.reachable]
        if self.partitioned and len(reachable) < len(self.replicas):
            print(f"[CP] READ REFUSED: partition detected → 503 unavailable")
            return None   # refuse to return potentially stale data
        result = reachable[0].read(key) if reachable else None
        print(f"[CP] READ OK: {key}={result}")
        return result

class APDatabase:
    """AP: availability over consistency. Serves stale reads during partition."""
    def __init__(self, replicas):
        self.replicas = replicas
        self.partitioned = False

    def write(self, key, value):
        # write to whichever replicas are reachable
        written_to = 0
        for r in self.replicas:
            if r.reachable:
                r.write(key, value)
                written_to += 1
        print(f"[AP] WRITE OK: {key}={value} on {written_to}/{len(self.replicas)} reachable replicas")
        return True   # always succeeds (even if only 1 replica acked)

    def read(self, key):
        # serve from nearest reachable replica — may be stale
        for r in self.replicas:
            if r.reachable:
                result = r.read(key)
                stale_note = " (possibly stale — partition active)" if self.partitioned else ""
                print(f"[AP] READ from {r.name}: {key}={result}{stale_note}")
                return result
        return None

# Setup: 3 replicas, one partitioned
r1 = Replica("DC1-primary")
r2 = Replica("DC2-replica")
r3 = Replica("DC3-replica", reachable=False)   # DC3 is partitioned

cp_db = CPDatabase([r1, r2, r3])
ap_db = APDatabase([r1, r2, r3])

print("=== Normal operation ===")
cp_db.write("user:1:email", "alice@old.com")
cp_db.read("user:1:email")

print("\n=== Network partition detected ===")
cp_db.partitioned = True
ap_db.partitioned = True

print("\nCP DB during partition:")
cp_db.write("user:1:email", "alice@new.com")   # refused
cp_db.read("user:1:email")                      # refused

print("\nAP DB during partition:")
ap_db.write("user:1:email", "alice@new.com")    # succeeds on reachable nodes
ap_db.read("user:1:email")                       # returns value (may be stale)

<a id='6'></a>
## 6. 🧩 Pattern 2: Consistent Hashing

---

```
PROBLEM:  Hash sharding: hash(key) % N → shard#
          When you add or remove a node, N changes → almost all keys reassigned
          → massive cache invalidation / data migration.

CONSISTENT HASHING SOLUTION:
  Map both nodes AND keys to the same circular hash ring [0, 2^32).
  A key belongs to the first node clockwise on the ring.

  Adding a node D between A and C:
    Only keys in arc (A, D] need to move from C to D.
    All other keys stay on their current node.
    Expected migration: 1/N of all keys (much better than N-1/N for simple hash).

  Removing node A:
    Only A's keys need to move to the next node.
    Expected migration: 1/N of all keys.

VIRTUAL NODES (vnodes):
  Problem: with few physical nodes, load may be uneven.
  Solution: each physical node occupies V positions on the ring.
  V=150: each physical node has 150 virtual positions → even distribution.
  Benefit: when a node is added/removed, its load is spread evenly
  across ALL other nodes (not just adjacent ones).

USED IN:
  Cassandra:   consistent hashing with vnodes for partition key routing
  DynamoDB:    consistent hashing for table partition assignment
  Memcached:   client-side consistent hashing for cache key routing
  Redis Cluster: hash slots (16384 slots, consistent hashing variant)
```

In [ ]:
# CONSISTENT HASHING IMPLEMENTATION

import hashlib
import bisect
from typing import List, Dict, Optional

class ConsistentHashRing:
    def __init__(self, virtual_nodes: int = 150):
        self.virtual_nodes = virtual_nodes   # vnodes per physical node
        self.ring: Dict[int, str] = {}       # hash position → node_id
        self.sorted_keys: List[int] = []     # sorted ring positions for bisect

    def _hash(self, key: str) -> int:
        return int(hashlib.md5(key.encode()).hexdigest(), 16) % (2**32)

    def add_node(self, node_id: str):
        for i in range(self.virtual_nodes):
            vnode_key = f"{node_id}:vnode:{i}"   # unique key for each virtual node
            pos = self._hash(vnode_key)
            self.ring[pos] = node_id
            bisect.insort(self.sorted_keys, pos)  # keep ring sorted
        print(f"Added {node_id} with {self.virtual_nodes} virtual nodes")

    def remove_node(self, node_id: str):
        for i in range(self.virtual_nodes):
            vnode_key = f"{node_id}:vnode:{i}"
            pos = self._hash(vnode_key)
            if pos in self.ring:
                del self.ring[pos]
                idx = bisect.bisect_left(self.sorted_keys, pos)
                self.sorted_keys.pop(idx)
        print(f"Removed {node_id}")

    def get_node(self, key: str) -> Optional[str]:
        if not self.ring:
            return None
        pos = self._hash(key)
        # find first node clockwise (first position >= key's hash)
        idx = bisect.bisect_left(self.sorted_keys, pos) % len(self.sorted_keys)
        ring_pos = self.sorted_keys[idx]
        return self.ring[ring_pos]

    def distribution(self, num_keys: int = 10000) -> Dict[str, int]:
        counts: Dict[str, int] = {}
        for i in range(num_keys):
            node = self.get_node(f"key_{i}")
            counts[node] = counts.get(node, 0) + 1
        return counts

# Demo: 3 nodes, add a 4th, compare key distribution
ring = ConsistentHashRing(virtual_nodes=150)
ring.add_node("NodeA")
ring.add_node("NodeB")
ring.add_node("NodeC")

print()
print("Distribution with 3 nodes (10,000 keys):")
dist3 = ring.distribution()
for node, count in sorted(dist3.items()):
    print(f"  {node}: {count:,} keys ({count/100:.1f}%)")

# Track which node each key is on BEFORE adding NodeD
test_keys = [f"user:{i}" for i in range(20)]
before = {k: ring.get_node(k) for k in test_keys}

print()
ring.add_node("NodeD")   # add 4th node

print("\nDistribution with 4 nodes:")
dist4 = ring.distribution()
for node, count in sorted(dist4.items()):
    print(f"  {node}: {count:,} keys ({count/100:.1f}%)")

# Count reassignments
after = {k: ring.get_node(k) for k in test_keys}
moved = sum(1 for k in test_keys if before[k] != after[k])
print(f"\nOf {len(test_keys)} sample keys: {moved} moved ({moved/len(test_keys):.0%})")
print(f"Expected ~{100//4}% movement (1/4 of keys reassigned when adding 4th node)")
print("Compare to simple hash: ~75% would move (hash % 3 → hash % 4 changes almost all)")

<a id='7'></a>
## 7. 🧩 Pattern 3: Replication Strategies

---

```
THREE REPLICATION TOPOLOGIES:

1. PRIMARY-REPLICA (single-leader):
   Client → Primary (writes)  Client → Any Replica (reads)
   Primary replicates to replicas asynchronously or synchronously.
   
   Sync replication:  write acks only after all replicas confirm
                       zero data loss on failover, but high write latency
   Async replication: write acks after primary writes locally
                       low write latency, but replica lag → stale reads
   Semi-sync (MySQL): at least one replica must ack → compromise

   Failover: if primary dies, elect a replica as new primary.
   Risk: async lag means elected replica may not have latest writes.

2. MULTI-PRIMARY (multi-leader):
   Each datacenter has its own primary. Primaries replicate to each other.
   Conflict resolution required (LWW, version vectors, CRDTs).
   Use case: multi-region writes with low cross-region latency requirement.

3. LEADERLESS (Dynamo-style):
   Any node accepts writes. Client writes to N nodes, waits for W acks.
   Reads: client reads from R nodes, takes the latest version.
   Quorum: W + R > N → strong consistency guaranteed.
   Read repair: when read detects stale replica, background sync.

REPLICA LAG PROBLEM (async replication):
  User writes post → immediately reads it → reads from lagging replica → sees nothing
  Fix: "read your own writes" — after a write, route reads to primary for N seconds
  Or: use a session token that forces reads from primary for that session
```

In [ ]:
# REPLICATION SIMULATION: primary-replica with sync vs async

from dataclasses import dataclass, field
from typing import List, Dict, Any, Optional
import time

@dataclass
class ReplicaNode:
    name: str
    is_primary: bool = False
    data: Dict[str, Any] = field(default_factory=dict)
    replication_lag_ms: int = 0   # simulated lag

    def apply_write(self, key, value):
        self.data[key] = value

    def read(self, key):
        return self.data.get(key)

class PrimaryReplicaCluster:
    def __init__(self, sync_replication: bool = True):
        self.sync = sync_replication
        self.primary = ReplicaNode("primary", is_primary=True)
        self.replicas = [
            ReplicaNode("replica1", replication_lag_ms=0),
            ReplicaNode("replica2", replication_lag_ms=500),  # 500ms behind
        ]
        mode = "SYNC" if sync_replication else "ASYNC"
        print(f"Cluster: 1 primary + {len(self.replicas)} replicas ({mode} replication)")

    def write(self, key, value):
        # always write to primary first
        self.primary.apply_write(key, value)

        if self.sync:
            # SYNC: wait for ALL replicas to confirm before acking client
            for r in self.replicas:
                r.apply_write(key, value)   # simulated: wait for each replica
            print(f"[SYNC WRITE] {key}={value} — confirmed on all {1+len(self.replicas)} nodes")
        else:
            # ASYNC: ack client immediately, replicate in background
            print(f"[ASYNC WRITE] {key}={value} — acked immediately (primary only)")
            print(f"  Background replication pending to {len(self.replicas)} replicas")
            # simulate: replica0 syncs immediately, replica1 has 500ms lag
            self.replicas[0].apply_write(key, value)  # fast replica syncs
            # self.replicas[1] NOT updated yet — simulating lag

    def read(self, key, from_replica_idx: int = None):
        if from_replica_idx is not None:
            node = self.replicas[from_replica_idx]
            value = node.read(key)
            lag_note = f" (lag={node.replication_lag_ms}ms)" if node.replication_lag_ms else ""
            print(f"[READ from {node.name}]{lag_note}: {key}={repr(value)}")
        else:
            value = self.primary.read(key)
            print(f"[READ from primary]: {key}={repr(value)}")
        return value

# Demo 1: Sync replication — strong consistency
print("=== SYNC REPLICATION ===")
sync_cluster = PrimaryReplicaCluster(sync_replication=True)
sync_cluster.write("user:1", {"name": "Alice"})
sync_cluster.read("user:1", from_replica_idx=1)   # lagging replica — but sync keeps it current

print()

# Demo 2: Async replication — replication lag problem
print("=== ASYNC REPLICATION ===")
async_cluster = PrimaryReplicaCluster(sync_replication=False)
async_cluster.write("user:1", {"name": "Bob"})
print("Immediately after write — reading from lagging replica:")
async_cluster.read("user:1", from_replica_idx=1)   # replica1 hasn't synced yet → None
async_cluster.read("user:1", from_replica_idx=0)   # replica0 synced immediately
async_cluster.read("user:1")                        # primary always has latest

print()
print("FIX — read-your-own-writes: route reads to primary for 1 second after write")
print("OR: use session consistency token to force primary reads for same session")

<a id='8'></a>
## 8. 🧩 Pattern 4: Partitioning / Sharding

---

```
PROBLEM:  Single database node can't hold all data or handle all requests.
          Shard the data across multiple nodes.

SHARDING STRATEGIES:

1. RANGE SHARDING:
   Shard1: user_id 0–999,999  Shard2: 1M–1.999M  Shard3: 2M+
   Pros: good for range queries (get all users 1000-2000 → one shard)
   Cons: hotspot risk — new user_ids cluster on the last shard (sequential inserts)

2. HASH SHARDING:
   shard = hash(user_id) % num_shards
   Pros: even distribution, no hotspots from sequential keys
   Cons: can't do range queries efficiently (user_id 1000-2000 → all shards)

3. CONSISTENT HASHING (see Pattern 2):
   Best of both: even distribution + minimal reshuffling on node change

4. DIRECTORY-BASED:
   Central lookup table: {user_id: shard_id}
   Pros: flexible, can rebalance any key to any shard
   Cons: lookup table is a bottleneck and single point of failure

HOTSPOT PROBLEM:
  Key: "user:1" gets 10,000 req/sec (celebrity account)
  All requests hit shard1 → overwhelmed while other shards sit idle
  Fixes:
    - Add random suffix: user:1:0, user:1:1, ..., user:1:9 → 10 shards handle it
    - Caching layer: Redis in front of shard for hot keys
    - Read replicas for hot keys only

SECONDARY INDEXES across shards:
  Problem: SELECT * WHERE email = 'alice@...' → must query ALL shards
  Fix1: Global secondary index (maintained separately, may be stale)
  Fix2: Document-based approach: store email → user_id mapping in separate table
  Fix3: Dual-write: write to both user_id shard AND email shard
```

In [ ]:
# SHARDING SIMULATION: range vs hash, hotspot demonstration

import hashlib
from collections import defaultdict
from typing import List

class RangeShardedTable:
    def __init__(self, num_shards: int):
        self.num_shards = num_shards
        self.shards = [defaultdict(dict) for _ in range(num_shards)]
        self.max_id_per_shard = 1_000_000  # 1M users per shard

    def shard_for(self, user_id: int) -> int:
        return min(user_id // self.max_id_per_shard, self.num_shards - 1)

    def write(self, user_id: int, data: dict):
        shard = self.shard_for(user_id)
        self.shards[shard][user_id] = data
        return shard

class HashShardedTable:
    def __init__(self, num_shards: int):
        self.num_shards = num_shards
        self.shards = [defaultdict(dict) for _ in range(num_shards)]

    def shard_for(self, user_id: int) -> int:
        return hash(str(user_id)) % self.num_shards

    def write(self, user_id: int, data: dict):
        shard = self.shard_for(user_id)
        self.shards[shard][user_id] = data
        return shard

# Simulate 100 new user registrations (sequential IDs: 2_000_000 to 2_000_099)
# These all start on the last shard in range sharding → hotspot!
range_table = RangeShardedTable(num_shards=3)
hash_table = HashShardedTable(num_shards=3)

range_shard_counts = [0, 0, 0]
hash_shard_counts = [0, 0, 0]

for uid in range(2_000_000, 2_000_100):  # 100 sequential new users
    s_range = range_table.write(uid, {"name": f"user_{uid}"})
    s_hash = hash_table.write(uid, {"name": f"user_{uid}"})
    range_shard_counts[s_range] += 1
    hash_shard_counts[s_hash] += 1

print("100 sequential new user registrations (uid 2M-2M+100):")
print("Range sharding distribution:")
for i, count in enumerate(range_shard_counts):
    bar = "█" * count
    print(f"  Shard {i}: {bar} ({count} users) {'← HOTSPOT' if count == 100 else ''}")

print("\nHash sharding distribution:")
for i, count in enumerate(hash_shard_counts):
    bar = "█" * count
    print(f"  Shard {i}: {bar} ({count} users)")

# Hotspot fix: random suffix for hot keys
print("\nHotspot fix — random suffix for celebrity user:1:")
import random
celebrity_shards = defaultdict(int)
for _ in range(1000):  # 1000 requests for celebrity
    suffix = random.randint(0, 9)  # spread across 10 virtual keys
    virtual_key = f"user:1:{suffix}"
    shard = hash(virtual_key) % 3
    celebrity_shards[shard] += 1
print("Requests per shard with suffix:", dict(sorted(celebrity_shards.items())))
print("Without suffix: all 1000 requests → shard", hash("user:1") % 3)

<a id='9'></a>
## 9. 🧩 Pattern 5: Distributed Consensus & Leader Election

---

```
PROBLEM:  In a cluster of N nodes, one must act as the "leader" (coordinator).
          If the leader dies, the cluster must elect a new one — without split-brain
          (two nodes both thinking they are leader → conflicting writes).

RAFT CONSENSUS (simplified):
  Three roles: Leader, Follower, Candidate

  Normal operation:
    Leader sends periodic heartbeats to all followers.
    Followers reset their election timer on each heartbeat.

  Leader failure detected:
    Follower's timer expires (no heartbeat received).
    Follower becomes Candidate, increments term, votes for itself.
    Candidate requests votes from other nodes.
    If quorum (majority = N/2+1) votes received → becomes new Leader.
    Immediately sends heartbeats to prevent other elections.

  SPLIT-BRAIN PREVENTION:
    Quorum rule: leader must have votes from majority.
    In a 5-node cluster: need 3 votes.
    If network splits into [2, 3]: only the group of 3 can elect a leader.
    The group of 2 cannot reach quorum → remains leaderless → no writes accepted.

  FENCING TOKENS:
    Each leader gets a monotonically increasing token (term number).
    All operations include the token. If a stale leader tries to write
    with an old token, the storage layer rejects it.
    Prevents zombie leader (thought-dead leader that reconnects) from writing.

USED IN:
  ZooKeeper:  Zab protocol (similar to Raft) — distributed lock, config
  etcd:       Raft — Kubernetes control plane coordination
  Kafka:      KRaft (Kafka Raft) replacing ZooKeeper for metadata management
```

In [ ]:
# RAFT LEADER ELECTION SIMULATION (simplified)

from dataclasses import dataclass, field
from typing import List, Optional, Set
import random

@dataclass
class RaftNode:
    node_id: int
    state: str = "follower"   # follower, candidate, leader
    current_term: int = 0
    voted_for: Optional[int] = None   # node_id this node voted for in current term
    alive: bool = True

    def request_vote(self, candidate_id: int, candidate_term: int) -> bool:
        if not self.alive:
            return False
        # grant vote if: haven't voted in this term OR already voted for this candidate
        if candidate_term > self.current_term:
            self.current_term = candidate_term
            self.state = "follower"
            self.voted_for = None
        if candidate_term == self.current_term and (
                self.voted_for is None or self.voted_for == candidate_id):
            self.voted_for = candidate_id
            return True   # vote granted
        return False      # vote denied (already voted for someone else)

class RaftCluster:
    def __init__(self, num_nodes: int):
        self.nodes = [RaftNode(i) for i in range(num_nodes)]
        self.quorum = num_nodes // 2 + 1   # majority threshold
        print(f"Cluster: {num_nodes} nodes, quorum={self.quorum}")

    def elect_leader(self, candidate_id: int) -> bool:
        candidate = self.nodes[candidate_id]
        candidate.state = "candidate"
        candidate.current_term += 1
        candidate.voted_for = candidate_id
        new_term = candidate.current_term

        votes = 1   # vote for self
        print(f"\nNode {candidate_id} starts election for term {new_term}:")

        for node in self.nodes:
            if node.node_id == candidate_id:
                continue
            granted = node.request_vote(candidate_id, new_term)
            print(f"  Node {node.node_id}: {'alive' if node.alive else 'DEAD'} → vote {'GRANTED' if granted else 'DENIED'}")
            if granted:
                votes += 1

        if votes >= self.quorum:
            candidate.state = "leader"
            print(f"  {votes}/{len(self.nodes)} votes → Node {candidate_id} becomes LEADER (term {new_term})")
            return True
        else:
            candidate.state = "follower"
            print(f"  {votes}/{len(self.nodes)} votes — NOT enough (need {self.quorum}) → election failed")
            return False

# Demo 1: normal election in 5-node cluster
print("=== NORMAL LEADER ELECTION (5 nodes) ===")
cluster = RaftCluster(5)
cluster.elect_leader(0)

# Demo 2: election when 2 nodes are dead (3 alive → quorum=3 → just barely)
print("\n=== ELECTION WITH 2 FAILED NODES ===")
cluster2 = RaftCluster(5)
cluster2.nodes[1].alive = False
cluster2.nodes[2].alive = False
cluster2.elect_leader(0)   # 3 alive nodes, quorum=3 → succeeds

# Demo 3: split-brain prevention — only 2 nodes in partition
print("\n=== SPLIT-BRAIN: only 2 of 5 nodes reachable ===")
cluster3 = RaftCluster(5)
cluster3.nodes[2].alive = False
cluster3.nodes[3].alive = False
cluster3.nodes[4].alive = False
cluster3.elect_leader(0)   # only 2 alive, quorum=3 → FAILS → no leader → no split-brain

<a id='10'></a>
## 10. 🗺️ The Distributed Systems Decision Map

```
REQUIREMENT                          PATTERN              SYSTEMS
──────────────────────────────────────────────────────────────────────────
Strong consistency required          CP, quorum W+R>N     ZooKeeper, HBase
High availability, tolerate stale    AP, eventual         Cassandra, DynamoDB
Even key distribution                Consistent hashing   Cassandra, Redis
Range queries on shards              Range partitioning   HBase, RDB
Hot key problem                      Virtual nodes / cache  Random suffix
Distributed locks / coordination     Consensus (Raft)     ZooKeeper, etcd
One writer, many readers             Primary-replica      MySQL, Postgres
Multi-region low-latency writes      Multi-primary        CockroachDB
──────────────────────────────────────────────────────────────────────────

QUORUM QUICK REFERENCE (N=3):
  W=2, R=2: W+R=4 > 3 → strong consistency (default choice)
  W=3, R=1: all writes must ack, reads from any replica
  W=1, R=1: eventual consistency — max throughput

PARTITION RESPONSE:
  Can't reach quorum?
    CP: return error, refuse to serve
    AP: serve from available replicas, note potential staleness
  When partition heals:
    AP: run anti-entropy / read repair to converge replicas
    CP: resume normal operation automatically

FENCING TOKEN PATTERN:
  Every leader operation includes: (term, sequence_number)
  Storage layer rejects any write with a lower term than current known term
  → zombie leaders (reconnected old leaders) can't corrupt state
```

<a id='11'></a>
## 11. 📋 Interview Cheat Sheet

### When to reach for each pattern:

| Requirement | Pattern | System |
|---|---|---|
| Financial/inventory consistency | CP (Raft/ZK) | ZooKeeper, etcd |
| High availability, stale ok | AP (eventual) | Cassandra, DynamoDB |
| Even sharding with easy scaling | Consistent hashing | Cassandra, Redis |
| Range queries on sorted key | Range sharding | HBase, RDS |
| One authoritative write path | Primary-replica | MySQL, Postgres |
| Distributed lock / leader | Consensus | ZooKeeper, etcd |

### Key formulas:

```
Strong consistency:    W + R > N
Quorum (majority):     floor(N/2) + 1
Consistent hashing:    ~1/N keys move when adding/removing a node
Simple hash sharding:  ~(N-1)/N keys move when adding a node
```

### CAP quick reference:

```
CP systems:  HBase, ZooKeeper, Spanner, PostgreSQL (single node)
AP systems:  Cassandra, DynamoDB, CouchDB, DNS, Kafka
THERE IS NO CA:  all production systems must tolerate partitions (P)
```

### Gotchas to not forget:

```
❌  "CA database" — doesn't exist; P is mandatory in distributed systems
✅  Frame as: "CP or AP choice during a partition"
❌  Async replication + immediate reads from replica → read-your-writes violation
✅  Route reads to primary for T seconds after write (read-your-writes guarantee)
❌  Range shard by auto-increment ID → all new writes hit last shard (hotspot)
✅  Hash shard auto-increment IDs, or use time-based prefix to spread load
❌  Leader election without fencing tokens → zombie leader can corrupt data
✅  Monotonic term/epoch numbers; storage layer rejects stale-term writes
❌  Forget virtual nodes in consistent hashing → uneven load on few physical nodes
✅  150+ vnodes per physical node for even distribution
```

<a id='12'></a>
## 12. 🗺️ Summary Map

```
           DISTRIBUTED SYSTEMS FUNDAMENTALS
                        │
          ┌─────────────┼─────────────┐
          │             │             │
       CAP             DATA        COORDINATION
     THEOREM        DISTRIBUTION       │
          │             │          Leader Election
     CP: strong    Consistent     Raft: quorum vote
     AP: available  Hashing       ZooKeeper: Zab
          │          Ring+vnodes  Fencing tokens
     CONSISTENCY  Range Sharding  etcd: Raft
     MODELS       Hash Sharding
          │       Directory-based
     eventual → linearizable

     REPLICATION
     ─────────────────────────────────────
     Primary-replica: simple, consistent
     Multi-primary:   multi-region writes
     Leaderless:      W+R>N quorum

CORE RULE: W + R > N  =  strong consistency
           (write + read quorums must overlap — always see latest)
```

---
*End of Distributed Systems Fundamentals Master Guide — Sean Edition*